# Pycox: DeepSurv Stratified by Batch


In [1]:
import os
os.getcwd()

'/home/nfs/dengy/dl-survival-miRNA/scripts/examples'

In [7]:
import os
import numpy as np
import torch
import torchtuples as tt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn_pandas import DataFrameMapper

os.chdir("../..")
from pss.pycox.models import CoxPH, CoxPHStratified, StratifiedDataset
from pss.pycox.evaluation.eval_surv import EvalSurv
from pss.utils import load_prepared_split #load_simulate_survival_data
from pss.run_models import DeepSurvPipeline, train_over_subsets

## *Test: Debugging*

### *Network code*

In [3]:
import numpy as np
import torch
from torch import Tensor

def cox_ph_loss_sorted(log_h: Tensor, events: Tensor, eps: float = 1e-7) -> Tensor:
    """Requires the input to be sorted by descending duration time.
    See DatasetDurationSorted.

    We calculate the negative log of $(\frac{h_i}{\sum_{j \in R_i} h_j})^d$,
    where h = exp(log_h) are the hazards and R is the risk set, and d is event.

    We just compute a cumulative sum, and not the true Risk sets. This is a
    limitati`on, but simple and fast.
    """
    if events.dtype is torch.bool:
        events = events.float()
    events = events.view(-1)
    log_h = log_h.view(-1)
    if events.sum() == 0:
        return log_h.sum() * 0.0  # update 08/11/25: safe dummy loss
    
    gamma = log_h.max()
    log_cumsum_h = log_h.sub(gamma).exp().cumsum(0).add(eps).log().add(gamma)
    return - log_h.sub(log_cumsum_h).mul(events).sum().div(events.sum())


def cox_ph_loss(log_h: Tensor, durations: Tensor, events: Tensor, eps: float = 1e-7) -> Tensor:
    """Loss for CoxPH model. If data is sorted by descending duration, see `cox_ph_loss_sorted`.

    We calculate the negative log of $(\frac{h_i}{\sum_{j \in R_i} h_j})^d$,
    where h = exp(log_h) are the hazards and R is the risk set, and d is event.

    We just compute a cumulative sum, and not the true Risk sets. This is a
    limitation, but simple and fast.
    """
    idx = durations.sort(descending=True)[1]
    events = events[idx]
    log_h = log_h[idx]
    return cox_ph_loss_sorted(log_h, events, eps)


####### [UPDATE] 07/07/2025
def stratified_cox_ph_loss(log_h: Tensor, durations: Tensor, events: Tensor, batch_indices: Tensor, eps: float = 1e-7) -> Tensor:
    """
    Stratified CoxPH loss that computes partial likelihood across batches.

    Arguments:
        log_h {torch.Tensor} -- Log hazard predictions for each instance.
        durations {torch.Tensor} -- Duration times for each instance.
        events {torch.Tensor} -- Event indicators (1 if event, 0 if censored).
        batch_indices {numpy array} -- Batch labels for each instance.
        eps {float} -- Small epsilon for numerical stability.

    Returns:
        torch.Tensor -- The total stratified negative log partial likelihood.
    """
    device = batch_indices.device
    unique_batches = torch.unique(batch_indices)
    losses = torch.zeros(len(unique_batches), device=device)
    n_valid_batch = 0
        
    for i, batch in enumerate(unique_batches):
        # Select data for the current batch
        mask = (batch_indices == batch)
        if mask.sum() == 0 or events[mask].sum() == 0:
            continue  # skip empty batch (added 08/11/25) or batch with no events
        
        # Sort by descending durations
        idx = torch.argsort(durations[mask], descending=True)
        
        events_batch = events[mask][idx]
        log_h_batch = log_h[mask][idx]
        if events_batch.sum() == 0:
            continue 
        
        losses[i] = cox_ph_loss_sorted(log_h_batch, events_batch, eps)
        n_valid_batch += 1
        
    if n_valid_batch == 0:
        return log_h.sum() * 0.0
    # print(n_valid_batch)
    return losses.sum()

In [4]:
# Create a PyTorch tensor
batch_indices = torch.tensor([1, 2, 2, 2, 2, 2, 3, 3, 4, 4], dtype=torch.float32)
durations = torch.tensor([169.5, 0.6, 12.3, 1.5, 3.8, 0.1, 0.1, 0.1, 0.6, 0.1], dtype=torch.float32)
events = torch.tensor([0, 1, 1, 1, 1, 1, 1, 1, 1, 1], dtype=torch.float32)
log_h = torch.tensor([-4.1238, 2.1188, -1.5863, -1.2239, 0.9088, 5.6637, 1.2920, 4.5356, 1.5392, 5.0004], dtype=torch.float32)

# Test out ufnction
device = batch_indices.device
unique_batches = torch.unique(batch_indices)
losses = torch.zeros(len(unique_batches), device=device)

for i, batch in enumerate(unique_batches):
    # i = 1
    # batch = 1
    # print(i)
    mask = (batch_indices == batch)
    if mask.sum() == 0:
        print(f"batch {batch} is empty")
        continue
    idx = torch.argsort(durations[mask], descending=True)
    # idx = durations[mask].sort(descending=True)[1]
    log_h_batch = log_h[mask][idx]
    events_batch = events[mask][idx]
    
    print(events_batch)
    if events_batch.sum() == 0:
        print(f"batch {int(batch)} has no events")
        continue
    
    losses[i] = cox_ph_loss_sorted(log_h_batch, events_batch, eps=1e-7)
    
losses.sum()

tensor([0.])
batch 1 has no events
tensor([1., 1., 1., 1., 1.])
tensor([1., 1.])
tensor([1., 1.])


tensor(0.5826)

In [5]:
stratified_cox_ph_loss(log_h, durations, events, batch_indices)

tensor(0.5826)

# Full process

In [64]:
batchNormType='BE10Asso00_normNone'
dataType='linear-moderate'
train_size=5000
random_state=42
time_col='time'
status_col='status'
batch_col='batch_id'
iter_i = 1

train_df, test_df = load_prepared_split(batchNormType=batchNormType,
                                        dataName=dataType,
                                        keep_batch=True,
                                        train_size=train_size,
                                        iter_i=iter_i)

print(f"Training data dimensions: {train_df.shape}")
print(f"Testing data dimensions:  {test_df.shape}")

Training data dimensions: (5000, 541)
Testing data dimensions:  (1000, 541)


In [65]:
hyperparameters = {
    "num_nodes": {"type": "categorical", "choices": [[64,64], [32,32], [16,16]]},
    "dropout": {"type": "float", "low": 0.1, "high": 0.5},
    "weight_decay": {"type": "float", "low": 1e-5, "high": 1e-2, "log": True},
    "learning_rate": {"type": "float", "low": 1e-4, "high": 1e-2, "log": True},
    "batch_size": {"type": "categorical", "choices": [128, 64, 32, 16]}
}
dl = DeepSurvPipeline(
    train_df=None, test_df=None,
    batchNormType=batchNormType,
    dataName=dataType,
    is_stratified=True,
    time_col=time_col,
    status_col=status_col,
    batch_col=batch_col,
    hyperparameters=hyperparameters
)

In [66]:
def _preprocess_data(df, mapper=None, fit_scaler=True):
    survival_cols = [time_col, status_col]
    covariate_cols = [col for col in df.columns if col not in survival_cols]
    # Transform features (miRNA expression)
    if fit_scaler or mapper is None:
        standardize = [([col], StandardScaler()) for col in covariate_cols]
        mapper = DataFrameMapper(standardize)
        x = mapper.fit_transform(df[covariate_cols]).astype('float32')
    else:
        x = mapper.transform(df[covariate_cols]).astype('float32')
    # Prepare labels (survival data)
    y = (df[time_col].values, df[status_col].values)
    
    return x, y, mapper

batch_ids_train = train_df[batch_col].to_numpy().reshape(-1)
batch_ids_test = test_df[[batch_col]].to_numpy().reshape(-1)

train_sub = train_df.drop(columns=[batch_col])
test_sub = test_df.drop(columns=[batch_col])

x_train, y_train, mapper = _preprocess_data(train_sub)
x_test, y_test, _ = _preprocess_data(test_sub, mapper=mapper, fit_scaler=False)

durations_train, events_train = y_train[0], y_train[1]
durations_test, events_test = y_test[0], y_test[1]

# Prepare data 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x_train = torch.from_numpy(x_train).to(device)
x_test = torch.from_numpy(x_test).to(device)

batch_ids_train = torch.from_numpy(batch_ids_train).long().to(device)
batch_ids_test   = torch.from_numpy(batch_ids_test).long().to(device)
durations_train = torch.from_numpy(durations_train).float().to(device)
durations_test = torch.from_numpy(durations_test).float().to(device)
events_train = torch.from_numpy(events_train).float().to(device)
events_test = torch.from_numpy(events_test).float().to(device)
y_train = (durations_train, events_train)
y_test = (durations_test, events_test)
        
# batch_ids_train = torch.from_numpy(batch_ids_train).long().to(device)
# batch_ids_test = torch.from_numpy(batch_ids_test).long().to(device)

print(device)
print(x_test.shape)           # Should be [n_samples, n_features]
print(durations_test.shape)   # Should be [n_samples]
print(events_test.shape)      # Should be [n_samples]
print(batch_ids_test.shape)   # Should be [n_samples]

cpu
torch.Size([1000, 538])
torch.Size([1000])
torch.Size([1000])
torch.Size([1000])


In [67]:
# x_train.shape
input_size = x_train.shape[1]
output_size = 1
num_nodes = [32,16]            # Default # layers & nodes
dropout = 0.2                    # Default dropout rate
learning_rate = 1e-3      # Default learning rate
batch_size = 128               # Default batch size
epochs = 500                      # Default number of epochs
batch_norm = True             # Default batch normalization
output_bias = True           # Default output bias
weight_decay = 1e-4         # Default weight decay
activation = torch.nn.ReLU

net = tt.practical.MLPVanilla(
    in_features=input_size,
    out_features=output_size,
    num_nodes=num_nodes,
    dropout=dropout, 
    batch_norm=batch_norm,
    activation=activation,
    output_bias=output_bias
).to(device)
optimizer = tt.optim.Adam(weight_decay=weight_decay, lr=learning_rate)

# Get default early stopping settings if not defined 
patience = 30
min_delta = 1e-3
callbacks = [tt.callbacks.EarlyStopping(patience=patience, min_delta=min_delta)]

### CoxPH

In [62]:
# CoxPH model
model = CoxPH(net, optimizer=optimizer)
log = model.fit(
    x_train, y_train,
    batch_size=batch_size,
    epochs=epochs,
    callbacks=callbacks, 
    verbose=True,
    val_data=(x_test, y_test),
    val_batch_size=batch_size
)

0:	[0s / 0s],		train_loss: 3.9673,	val_loss: 3.7043
1:	[0s / 0s],		train_loss: 3.7355,	val_loss: 3.5769
2:	[0s / 0s],		train_loss: 3.6178,	val_loss: 3.5161
3:	[0s / 0s],		train_loss: 3.5591,	val_loss: 3.5114
4:	[0s / 0s],		train_loss: 3.5294,	val_loss: 3.4877
5:	[0s / 0s],		train_loss: 3.5231,	val_loss: 3.5193
6:	[0s / 0s],		train_loss: 3.5005,	val_loss: 3.4857
7:	[0s / 0s],		train_loss: 3.4976,	val_loss: 3.4783
8:	[0s / 0s],		train_loss: 3.5043,	val_loss: 3.4883
9:	[0s / 0s],		train_loss: 3.5043,	val_loss: 3.4770
10:	[0s / 0s],		train_loss: 3.5542,	val_loss: 3.4943
11:	[0s / 0s],		train_loss: 3.5419,	val_loss: 3.4760
12:	[0s / 0s],		train_loss: 3.5104,	val_loss: 3.5035
13:	[0s / 0s],		train_loss: 3.4897,	val_loss: 3.4846
14:	[0s / 0s],		train_loss: 3.5069,	val_loss: 3.4960
15:	[0s / 0s],		train_loss: 3.4648,	val_loss: 3.4949
16:	[0s / 0s],		train_loss: 3.4669,	val_loss: 3.4773
17:	[0s / 0s],		train_loss: 3.4859,	val_loss: 3.4965
18:	[0s / 0s],		train_loss: 3.4553,	val_loss: 3.4628
19:

In [63]:
# ==================== Evaluation ====================
_ = model.compute_baseline_hazards(input=x_train, target=(durations_train, events_train))

# Convert torch tensors back to numpy objects for evaluation
x_train_np = x_train.detach().cpu().numpy()
x_test  = x_test.detach().cpu().numpy()
durations_train = durations_train.detach().cpu().numpy()
durations_test  = durations_test.detach().cpu().numpy()
events_train    = events_train.detach().cpu().numpy()
events_test     = events_test.detach().cpu().numpy()

# Initialize EvalSurv objects 
tr_surv  = model.predict_surv_df(x_train)
te_surv = model.predict_surv_df(x_test)
tr_ev = EvalSurv(tr_surv, durations_train, events_train, censor_surv='km')
te_ev = EvalSurv(te_surv, durations_test, events_test, censor_surv='km')

# Concordance index ----------------
tr_c_index  = tr_ev.concordance_td() 
te_c_index = te_ev.concordance_td() 

tr_c_index, te_c_index

((0.8268978232491716, 10755480.0), (0.8127147844947903, 437066.0))

### Stratified CoxPH

In [ ]:
# train_dataset = StratifiedDataset(x_train, durations_train, events_train, batch_ids_train)
# train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# test_dataset = torch.utils.data.TensorDataset(x_test, durations_test, events_test, batch_ids_test)
# test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

# # # Access batches
# # for idx, (inputs, durations, events, batch_ids) in enumerate(train_loader):
# #     print(f"Batch {idx + 1}:")
# #     print("batch ids:", batch_ids)
# #     # print("Time:", durations)
# #     print("Events:", events)
# #     print()
    
# ## Test
# for idx, (inputs, durations, events, batch_ids) in enumerate(test_loader):
#     print(f"Batch {idx + 1}:")
#     print("batch ids:", batch_ids)
#     # print("Time:", durations)
#     print("Events:", events)
#     print()

In [ ]:
# train_dataset = torch.utils.data.TensorDataset(x_train, durations_train, events_train, batch_ids_train)
# test_dataset   = torch.utils.data.TensorDataset(x_test, durations_test, events_test, batch_ids_test)
train_dataset = StratifiedDataset(x_train, durations_train, events_train, batch_ids_train)
test_dataset = StratifiedDataset(x_test, durations_test, events_test, batch_ids_test)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

for xb, db, eb, bb in test_loader:
    print("VAL batch total events:", int(eb.sum().item()),
          "| per-stratum:", {int(s): int(eb[bb==s].sum().item()) for s in bb.unique().tolist()})

VAL batch total events: 94 | per-stratum: {1: 50, 2: 44}
VAL batch total events: 95 | per-stratum: {2: 4, 3: 51, 4: 40}
VAL batch total events: 103 | per-stratum: {4: 10, 5: 54, 6: 39}
VAL batch total events: 99 | per-stratum: {6: 12, 7: 57, 8: 30}
VAL batch total events: 104 | per-stratum: {8: 21, 9: 52, 10: 31}
VAL batch total events: 98 | per-stratum: {10: 23, 11: 51, 12: 24}
VAL batch total events: 93 | per-stratum: {12: 20, 13: 52, 14: 21}
VAL batch total events: 79 | per-stratum: {14: 28, 15: 51}


In [69]:
import time

# Stratified CoxPH model
model = CoxPHStratified(net, optimizer=optimizer)
model.metrics = {'val_loss': model.loss}
start = time.time() # Record iteration start time
log = model.fit_dataloader(
    train_loader,
    epochs=epochs,
    callbacks=callbacks,
    verbose=True,
    val_dataloader=test_loader  # optional for now
)
stop = time.time() # Record time when training finished
duration = round(stop - start, 2)
print(f"Training time: {duration}")

0:	[0s / 0s],		train_loss: 20.7019,	val_loss: 7.1431
1:	[0s / 0s],		train_loss: 17.8641,	val_loss: 6.6906
2:	[0s / 0s],		train_loss: 16.9556,	val_loss: 6.5517
3:	[0s / 0s],		train_loss: 16.2086,	val_loss: 6.5970
4:	[0s / 0s],		train_loss: 15.9425,	val_loss: 6.5407
5:	[0s / 1s],		train_loss: 15.6534,	val_loss: 6.4848
6:	[0s / 1s],		train_loss: 15.5020,	val_loss: 6.4628
7:	[0s / 1s],		train_loss: 15.4775,	val_loss: 6.4974
8:	[0s / 1s],		train_loss: 15.6264,	val_loss: 6.4805
9:	[0s / 1s],		train_loss: 15.1718,	val_loss: 6.4457
10:	[0s / 1s],		train_loss: 15.1951,	val_loss: 6.4510
11:	[0s / 2s],		train_loss: 15.5231,	val_loss: 6.4875
12:	[0s / 2s],		train_loss: 15.1894,	val_loss: 6.4681
13:	[0s / 2s],		train_loss: 15.3144,	val_loss: 6.4707
14:	[0s / 2s],		train_loss: 15.3350,	val_loss: 6.4489
15:	[0s / 2s],		train_loss: 15.0921,	val_loss: 6.4842
16:	[0s / 2s],		train_loss: 15.0821,	val_loss: 6.4735
17:	[0s / 3s],		train_loss: 14.9173,	val_loss: 6.4913
18:	[0s / 3s],		train_loss: 15.2837,	v

### Evaluation

#### *Stratified C-index* 

In [70]:
# ==================== Evaluation ====================
# Convert torch tensors back to numpy objects for evaluation
durations_train_np = durations_train.detach().cpu().numpy()
durations_test_np  = durations_test.detach().cpu().numpy()
events_train_np    = events_train.detach().cpu().numpy()
events_test_np     = events_test.detach().cpu().numpy()
batch_ids_train_np = batch_ids_train.detach().cpu().numpy()
batch_ids_test_np   = batch_ids_test.detach().cpu().numpy()

# Compute baseline hazards (per-batch)
baseline_hazards_strata = model.compute_baseline_hazards(input=x_train, target=(durations_train, events_train), batch_ids=batch_ids_train_np)

# Initialize EvalSurv objects 
tr_surv  = model.predict_surv_df(x_train, batch_ids = batch_ids_train_np)
te_surv = model.predict_surv_df(x_test, batch_ids = batch_ids_test_np)
tr_ev = EvalSurv(tr_surv, durations_train_np, events_train_np, censor_surv='km')
te_ev = EvalSurv(te_surv, durations_test_np, events_test_np, censor_surv='km')

# Concordance index ----------------
tr_strat_c_index  = tr_ev.stratified_concordance_td(batch_indices=batch_ids_train_np) 
te_strat_c_index = te_ev.stratified_concordance_td(batch_indices=batch_ids_test_np) 

print(tr_strat_c_index, te_strat_c_index)

0.8278145139869427 0.8083608552059794


In [71]:
all_times = None
per_batch_surv = {}
for b, haz in baseline_hazards_strata.items():
    bch = haz.cumsum()   # H_0,b(t)
    per_batch_surv[b] = bch   
    all_times = bch.index if all_times is None else all_times.union(bch.index)
all_times = all_times.sort_values()
all_times

Index([0.0016515491297468543, 0.0030213501304388046,  0.003553541377186775, 0.0036848702002316713,  0.005831246729940176,
        0.006474142428487539,   0.00816679559648037,  0.010230381041765213,  0.011380677111446857,  0.011532609350979328,
       ...
             196.86962890625,     197.3412322998047,    197.49827575683594,     198.0352020263672,    198.53106689453125,
          199.19259643554688,    199.27955627441406,    199.36724853515625,       199.43603515625,    199.99974060058594],
      dtype='float32', name='duration', length=5000)

In [72]:
expg = np.exp(model.predict(x_train, 8224, True, True, num_workers=0)).reshape(-1)

In [ ]:
out = np.empty((len(all_times), x_train.shape[0]), dtype=float)
for b in np.unique(batch_ids_train):
    idx = np.where(batch_ids_train == b)[0]
    bch = per_batch_surv[b]
    # align H0b onto union grid with forward-fill
    H0b = bch.reindex(all_times, method='ffill').fillna(0.0).values
    # survival = exp(- H0_b(t) * exp(g(x)))
    out[:, idx] = np.exp(- np.outer(H0b, expg[idx]))

In [74]:
all_times

Index([0.0016515491297468543, 0.0030213501304388046,  0.003553541377186775, 0.0036848702002316713,  0.005831246729940176,
        0.006474142428487539,   0.00816679559648037,  0.010230381041765213,  0.011380677111446857,  0.011532609350979328,
       ...
             196.86962890625,     197.3412322998047,    197.49827575683594,     198.0352020263672,    198.53106689453125,
          199.19259643554688,    199.27955627441406,    199.36724853515625,       199.43603515625,    199.99974060058594],
      dtype='float32', name='duration', length=5000)

In [75]:
out

array([[1.00000000e+00, 1.00000000e+00, 1.00000000e+00, ..., 1.00000000e+00, 1.00000000e+00, 1.00000000e+00],
       [1.00000000e+00, 1.00000000e+00, 1.00000000e+00, ..., 1.00000000e+00, 1.00000000e+00, 1.00000000e+00],
       [1.00000000e+00, 1.00000000e+00, 1.00000000e+00, ..., 1.00000000e+00, 1.00000000e+00, 1.00000000e+00],
       ...,
       [3.37574035e-02, 3.98165686e-03, 8.97872448e-01, ..., 2.96045698e-07, 5.35372848e-08, 1.71730220e-01],
       [3.37574035e-02, 3.98165686e-03, 8.97872448e-01, ..., 2.96045698e-07, 5.35372848e-08, 1.71730220e-01],
       [3.37574035e-02, 3.98165686e-03, 8.97872448e-01, ..., 2.96045698e-07, 5.35372848e-08, 1.71730220e-01]])

#### *One-batch C-index* 

In [76]:
baseline_hazards_1batch = model.compute_baseline_hazards(input=x_train, target=(durations_train, events_train))

# Initialize EvalSurv objects 
tr_surv  = model.predict_surv_df(x_train, baseline_hazards_=baseline_hazards_1batch)
te_surv = model.predict_surv_df(x_test, baseline_hazards_=baseline_hazards_1batch)
tr_ev = EvalSurv(tr_surv, durations_train_np, events_train_np, censor_surv='km')
te_ev = EvalSurv(te_surv, durations_test_np, events_test_np, censor_surv='km')

# Concordance index (non-stratified) ----------------
tr_c_index, _  = tr_ev.concordance_td() 
te_c_index, _ = te_ev.concordance_td() 
print(tr_c_index, te_c_index)

0.8207613700178885 0.8128005838935081


In [78]:
# Manual test
from pss.pycox.evaluation.concordance import concordance_td
from pss.pycox.evaluation import ipcw

batch_indices = batch_ids_train.detach().cpu().numpy() if not isinstance(batch_ids_train, np.ndarray) else batch_ids_train
batches = np.unique(batch_indices)
c_index_ls , n_pairs_ls = np.zeros(len(batches)), np.zeros(len(batches))

for i, batch in enumerate(batches):
    # Filter data by batch
    mask = (batch_indices == batch)
    if mask.sum() == 0:
        continue  # skip empty batch
    batch_durations = durations_train_np[mask]
    batch_events = events_train_np[mask]
    batch_surv = tr_ev.surv.iloc[:, mask]
    if batch_events.sum() == 0:
        continue
    
    # Compute concordance for the current batch
    c_index_batch, n_pairs_batch = concordance_td(
        batch_durations, batch_events, batch_surv.values,
        tr_ev.idx_at_times(batch_durations), method='adj_antolini'
    )
    print(n_pairs_batch)
    n_pairs_ls[i] = n_pairs_batch
    # n_events_ls[i] = batch_events.sum()
    c_index_ls[i] = c_index_batch
    
print("Final score: %f" % (np.sum(c_index_ls*n_pairs_ls) / np.sum(n_pairs_ls) if np.sum(n_pairs_ls) > 0 else float('nan')))

for e, c in zip(n_pairs_ls, c_index_ls):
    print(f"{int(e)} comparable pairs: {round(c,3)}")

48311.0
47984.0
47258.0
47133.0
47472.0
46095.0
47901.0
48081.0
49820.0
48245.0
48384.0
46327.0
48726.0
47620.0
45953.0
Final score: 0.827815
48311 comparable pairs: 0.825
47984 comparable pairs: 0.844
47258 comparable pairs: 0.816
47133 comparable pairs: 0.833
47472 comparable pairs: 0.806
46095 comparable pairs: 0.83
47901 comparable pairs: 0.833
48081 comparable pairs: 0.833
49820 comparable pairs: 0.822
48245 comparable pairs: 0.825
48384 comparable pairs: 0.822
46327 comparable pairs: 0.836
48726 comparable pairs: 0.842
47620 comparable pairs: 0.826
45953 comparable pairs: 0.823


#### *Test: Stratified Integrated Brier score*

In [ ]:
# Integrated Brier score -----------
min_surv = np.ceil(max(np.min(durations_train), np.min(durations_test)))
max_surv = np.floor(min(np.max(durations_train), np.max(durations_test)))
times = np.linspace(min_surv, max_surv, 20)

tr_brier  = tr_ev.integrated_brier_score(time_grid=times) 
te_brier =  te_ev.integrated_brier_score(time_grid=times)

print(tr_brier, te_brier)

In [ ]:
tr_strat_brier  = tr_ev.stratified_integrated_brier_score(time_grid=times, batch_indices=batch_ids_train) 
te_strat_brier =  te_ev.stratified_integrated_brier_score(time_grid=times, batch_indices=batch_ids_test)
print(tr_strat_brier, te_strat_brier)

In [ ]:
# print(tr_ev.surv.values.shape) 
# print(tr_ev.censor_surv.surv.values.shape)
# print(tr_ev.index_surv.shape)
# print(tr_ev.censor_surv.index_surv.shape) 
# print(tr_ev.steps)
# print(tr_ev.censor_surv.steps)

In [ ]:
batch_indices = batch_ids_train.detach().numpy() if not isinstance(batch_ids_train, np.ndarray) else batch_ids_train
batches = np.unique(batch_indices)
brier_ls, n_events_ls = np.zeros(len(batches)), np.zeros(len(batches))

for i, batch in enumerate(batches):
    # Filter data by batch
    mask = (batch_indices == batch)
    if mask.sum() == 0:
        continue  # skip empty batch
    batch_durations = durations_train[mask]
    batch_events = events_train[mask]
    batch_surv = tr_ev.surv.iloc[:, mask]
    if batch_events.sum() == 0:
        continue
    batch_surv_values = tr_ev.surv.values[:, mask]
    batch_censor_surv_values = tr_ev.censor_surv.surv.values[:, mask] 
    # batch_index_surv = tr_ev.index_surv[mask]
    # batch_censor_index_surv = tr_ev.censor_surv.index_surv[mask]
    
    # Compute integrated brier score for the current batch
    brier_batch = ipcw.integrated_brier_score(times, batch_durations, batch_events, 
                                    batch_surv_values, batch_censor_surv_values, 
                                    tr_ev.index_surv, tr_ev.censor_surv.index_surv, np.inf, 
                                    tr_ev.steps, tr_ev.censor_surv.steps)
    n_events_ls[i] = batch_events.sum()
    brier_ls[i] = brier_batch
    
print("Final score: %f\n" % (np.sum(brier_ls*n_events_ls) / np.sum(n_events_ls) if np.sum(n_events_ls) > 0 else float('nan')))

for e, c in zip(n_events_ls, brier_ls):
    print(f"{int(e)} events: {round(c,3)}")

# Pipeline Test

In [8]:
batchNormType='BE10Asso00_normNone'
dataType='linear-moderate'
train_size=5000
random_state=42
time_col='time'
status_col='status'
batch_col='batch_id'
iter_i = 1

train_df, test_df = load_prepared_split(batchNormType=batchNormType,
                                        dataName=dataType,
                                        keep_batch=True,
                                        train_size=train_size,
                                        iter_i=iter_i)

print(f"Training data dimensions: {train_df.shape}")
print(f"Testing data dimensions:  {test_df.shape}")

Training data dimensions: (5000, 541)
Testing data dimensions:  (1000, 541)


In [11]:
# import optuna
# # optuna.get_all_study_names("sqlite:///deepsurv-torch-hp-log.db")]
# optuna.delete_study(storage="sqlite:///deepsurv-torch-hp-log.db",
#                     study_name='BE10Asso00_normNone-linear-moderate-stratified-deepsurv-torch-5000')

In [ ]:
hyperparameters = {
    "num_nodes": {"type": "categorical", "choices": [
        # [128],[64],[32],
        # [128, 64],
        # [64, 32],
        # [32, 16],
        # [64, 64, 32]
        "128", "64", "32", "32-16", "64-32", "128-64", "64-64-32", "32-32-16"
    ]},
    "dropout": {"type": "float", "low": 0.1, "high": 0.5},
    "weight_decay": {"type": "float", "low": 1e-6, "high": 1e-2, "log": True},
    "learning_rate": {"type": "float", "low": 1e-5, "high": 5e-3, "log": True},
    "batch_size": {"type": "categorical", "choices": [256, 128, 64, 32]}
}

ds = DeepSurvPipeline(
    train_df=None, test_df=None,
    batchNormType=batchNormType,
    dataName=dataType,
    is_stratified=False,
    time_col=time_col,
    status_col=status_col,
    batch_col=batch_col,
    hyperparameters=hyperparameters,
    storage_url = "sqlite:///deepsurv-torch-hp-log.db"
)

# optuna.logging.disable_default_handler()
results = train_over_subsets(
    pipeline = ds,
    subset_sizes=[1000],#subset_sizes, 
    runs_per_size=[10],#runs_per_size, 
    splits_per_size=[10],#splits_per_size,
    trials_per_size=[30],#trails_per_size,
    is_tune=True, 
    is_save=False, 
    n_jobs=1,
    trial_threshold=30                              
)
results

Running for training size N=1000...


[I 2026-03-19 09:55:52,094] A new study created in RDB with name: BE10Asso00_normNone-linear-moderate-deepsurv-torch-1000


⚠️No completed trials in Optuna study 'BE10Asso00_normNone-linear-moderate-deepsurv-torch-1000'. Start hyperparameter tuning...


[W 2026-03-19 09:56:14,297] Trial 26 failed with parameters: {'num_nodes': '128', 'dropout': 0.4354219195143624} because of the following error: StorageInternalError('An exception is raised during the commit. This typically happens due to invalid data in the commit, e.g. exceeding max length. ').
Traceback (most recent call last):
  File "/home/nfs/dengy/dl-surv/lib/python3.10/site-packages/sqlalchemy/engine/base.py", line 1967, in _exec_single_context
    self.dialect.do_execute(
  File "/home/nfs/dengy/dl-surv/lib/python3.10/site-packages/sqlalchemy/engine/default.py", line 951, in do_execute
    cursor.execute(statement, parameters)
sqlite3.OperationalError: database is locked

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/home/nfs/dengy/dl-surv/lib/python3.10/site-packages/optuna/storages/_rdb/storage.py", line 78, in _create_scoped_session
    session.commit()
  File "/home/nfs/dengy/dl-surv/lib/python3.10/site-pa

# ==== Archive ====

In [ ]:
# prepare data
folder = 'linear'
keywords = ['moderate', "latest", 'RW']

train_df, test_df = load_simulate_survival_data(folder=folder, keywords=keywords, test_size=0.2)

train_df.head()

## Feature transforms


In [ ]:
survival_cols = ['time', 'status']

In [ ]:
tr_df, val_df = train_test_split(train_df, 
                                test_size=0.2,
                                shuffle=True, random_state=42,
                                stratify=train_df['status'])

# Transform data
covariate_cols = [col for col in train_df.columns if col not in survival_cols]
standardize = [([col], StandardScaler()) for col in covariate_cols]
leave = [(col, None) for col in survival_cols]
x_mapper = DataFrameMapper(standardize)

# gene expression data
x_train = x_mapper.fit_transform(tr_df[covariate_cols]).astype('float32')
x_val = x_mapper.fit_transform(val_df[covariate_cols]).astype('float32')
x_test = x_mapper.transform(test_df[covariate_cols]).astype('float32')

# prepare labels
get_target = lambda df: (df['time'].values, df['status'].values)
y_train = get_target(tr_df)
y_val = get_target(val_df)
t_test, e_test = get_target(test_df)
val = x_val, y_val

## Neural net

We create a simple MLP with two hidden layers, ReLU activations, batch norm and dropout. 
Here, we just use the `torchtuples.practical.MLPVanilla` net to do this.


In [ ]:
in_features = x_train.shape[1]
num_nodes = [32, 16]
out_features = 1
batch_norm = True
dropout = 0.2
output_bias = True

net = tt.practical.MLPVanilla(in_features, num_nodes, out_features, batch_norm,
                            dropout, output_bias=output_bias)

## Training the model

To train the model we need to define a `torch.optim` optimizer; here we instead use one from `tt.optim` as it has some added functionality.
We use the `Adam` optimizer and set the desired learning rate with `model.lr_finder`.

In [ ]:
optimizer = tt.optim.Adam(weight_decay=0.01)

be_model = CoxPHStratified(net, optimizer)

# we  set it manually to 0.001
be_model.optimizer.set_lr(1e-3)

We include the `EarlyStopping` callback to stop training when the validation loss stops improving. After training, this callback will also load the best performing model in terms of validation loss.

In [ ]:
%%time
batch_size = 64
epochs = 500
callbacks = [tt.callbacks.EarlyStopping(patience=20, min_delta=5e-2)]
verbose = True

batch_indices = np.ones(len(y_train[1]))
log = be_model.fit(x_train, y_train,
                batch_indices,
                batch_size,
                epochs,
                callbacks, 
                verbose=verbose,
                val_data=val, val_batch_size=batch_size
                )